In [1]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from scipy.stats import poisson
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings

# 경고 메시지 숨기기
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# 1. Advanced Feature Engineering (Dynamic Elo)
# ==========================================
class AdvancedElo:
    def __init__(self, k=20, base=1500):
        self.k = k
        self.ratings = {}
        self.base = base

    def get_rating(self, team):
        return self.ratings.get(team, self.base)

    def expected_result(self, loc, awc):
        dr = loc - awc
        return 1 / (10 ** (-dr / 400) + 1)

    def update_rating(self, home_team, away_team, home_goals, away_goals, tournament_weight=1.0):
        home_rating = self.get_rating(home_team)
        away_rating = self.get_rating(away_team)

        goal_diff = abs(home_goals - away_goals)
        if goal_diff <= 1:
            G = 1.0
        elif goal_diff == 2:
            G = 1.5
        else:
            G = (11 + goal_diff) / 8.0

        home_expected = self.expected_result(home_rating + 100, away_rating)
        away_expected = self.expected_result(away_rating, home_rating + 100)

        if home_goals > away_goals:
            home_actual, away_actual = 1, 0
        elif home_goals < away_goals:
            home_actual, away_actual = 0, 1
        else:
            home_actual, away_actual = 0.5, 0.5

        home_new = home_rating + self.k * G * tournament_weight * (home_actual - home_expected)
        away_new = away_rating + self.k * G * tournament_weight * (away_actual - away_expected)

        self.ratings[home_team] = home_new
        self.ratings[away_team] = away_new

# ==========================================
# 2. Model Training (LightGBM + Optuna Optimization)
# ==========================================
def optimize_and_train(X, y, n_trials=50):
    X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)
    train_data = lgb.Dataset(X_train, label=y_train)
    valid_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)

    def objective(trial):
        params = {
            'objective': 'poisson',
            'metric': 'poisson',
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 15, 127),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
            'feature_pre_filter': False, 
            'verbose': -1,
            'seed': 42
        }
        
        gbm = lgb.train(
            params,
            train_data,
            valid_sets=[valid_data],
            num_boost_round=2000,
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )
        return gbm.best_score['valid_0']['poisson']

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials)
    
    best_params = study.best_params
    best_params['objective'] = 'poisson'
    best_params['feature_pre_filter'] = False
    best_params['verbose'] = -1
    best_params['seed'] = 42
    
    final_model = lgb.train(
        best_params, 
        lgb.Dataset(X, label=y), 
        num_boost_round=1000
    )
    
    return final_model

def train_lgbm_models(df_train, n_trials=50):
    features = [
        'home_elo', 'away_elo', 'elo_diff', 'home_avg_scored', 'away_avg_conceded',
        'away_avg_scored', 'home_avg_conceded', 'home_form_5', 'away_form_5', 'is_neutral'
    ]
    
    X = df_train[features]
    y_home = df_train['home_score']
    y_away = df_train['away_score']

    print("\n[1/2] 홈팀 득점 예측 모델 튜닝 중...")
    lgb_home = optimize_and_train(X, y_home, n_trials=n_trials)
    
    print("\n[2/2] 원정팀 득점 예측 모델 튜닝 중...")
    lgb_away = optimize_and_train(X, y_away, n_trials=n_trials)

    return lgb_home, lgb_away, features

# ==========================================
# 3. Monte Carlo Simulator Engine
# ==========================================
def simulate_match(lambda_home, lambda_away, n_simulations=10000):
    home_goals_sim = poisson.rvs(mu=lambda_home, size=n_simulations)
    away_goals_sim = poisson.rvs(mu=lambda_away, size=n_simulations)
    
    home_wins = np.sum(home_goals_sim > away_goals_sim)
    draws = np.sum(home_goals_sim == away_goals_sim)
    away_wins = np.sum(home_goals_sim < away_goals_sim)
    
    prob_h = home_wins / n_simulations
    prob_d = draws / n_simulations
    prob_a = away_wins / n_simulations
    
    scores = [f"{h}-{a}" for h, a in zip(home_goals_sim, away_goals_sim)]
    most_frequent_score = max(set(scores), key=scores.count)
    pred_h_score, pred_a_score = map(int, most_frequent_score.split('-'))
    
    return prob_h, prob_d, prob_a, pred_h_score, pred_a_score

def predict_tournament(test_matches, lgb_home, lgb_away, features, n_simulations=10000):
    X_test = test_matches[features]
    
    test_matches['lambda_home'] = lgb_home.predict(X_test)
    test_matches['lambda_away'] = lgb_away.predict(X_test)
    
    results = []
    print(f"\n🏃‍♂️ 각 경기당 {n_simulations:,}번의 몬테카를로 시뮬레이션을 실행합니다...")
    
    for idx, row in tqdm(test_matches.iterrows(), total=len(test_matches)):
        p_home, p_draw, p_away, pred_h, pred_a = simulate_match(row['lambda_home'], row['lambda_away'], n_simulations)
        
        if (p_home + p_away) > 0:
            team1_prob_final = p_home + p_draw * (p_home / (p_home + p_away))
            team2_prob_final = p_away + p_draw * (p_away / (p_home + p_away))
        else:
            team1_prob_final = 0.5
            team2_prob_final = 0.5
            
        if pred_h == pred_a:
            if team1_prob_final > team2_prob_final:
                pred_h += 1 
            elif team2_prob_final > team1_prob_final:
                pred_a += 1

        match_type = row.get('type', 'group')
        if pd.isna(match_type) or match_type == '':
            match_type = 'group'

        results.append({
            'team1': row['home_team'],
            'team2': row['away_team'],
            'team1_score': int(pred_h),
            'team2_score': int(pred_a),
            'team1_prob': round(team1_prob_final, 4),
            'team2_prob': round(team2_prob_final, 4),
            'type': match_type
        })
        
    output_cols = ['team1', 'team2', 'team1_score', 'team2_score', 'team1_prob', 'team2_prob', 'type']
    return pd.DataFrame(results, columns=output_cols)

# ==========================================
# 4. Execution Pipeline
# ==========================================
if __name__ == "__main__":
    print("🚀 월드컵 대진표 예측 및 포맷 변환 통합 파이프라인 시작...\n")
    
    file_path = "historical_results.csv" 
    output_path = "submission_consensus_formatted.csv"
    
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"❌ '{file_path}' 파일을 찾을 수 없습니다. 파일이 스크립트와 같은 경로에 있는지 확인해 주세요.")

    # [수정] 파일 안의 유럽 특수문자가 깨지지 않도록 인코딩 자동 보정 처리
    try:
        df = pd.read_csv(file_path, encoding='cp949')
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding='latin1')
    
    df['home_score'] = pd.to_numeric(df['home_score'], errors='coerce')
    df['away_score'] = pd.to_numeric(df['away_score'], errors='coerce')

    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    df['is_neutral'] = df['neutral'].apply(lambda x: 1 if str(x).upper() == 'TRUE' else 0)

    elo_system = AdvancedElo()
    home_elos, away_elos = [], []
    team_stats = {}
    h_avg_s, h_avg_c, a_avg_s, a_avg_c, h_form, a_form = [], [], [], [], [], []

    for idx, row in df.iterrows():
        ht, at = row['home_team'], row['away_team']
        
        home_elos.append(elo_system.get_rating(ht))
        away_elos.append(elo_system.get_rating(at))
        
        for t in (ht, at):
            if t not in team_stats:
                team_stats[t] = {'scored': [], 'conceded': [], 'pts': []}
                
        if len(team_stats[ht]['scored']) > 0:
            h_avg_s.append(np.mean(team_stats[ht]['scored'][-10:]))
            h_avg_c.append(np.mean(team_stats[ht]['conceded'][-10:]))
            h_form.append(sum(team_stats[ht]['pts'][-5:]))
        else:
            h_avg_s.append(1.0); h_avg_c.append(1.0); h_form.append(0)
            
        if len(team_stats[at]['scored']) > 0:
            a_avg_s.append(np.mean(team_stats[at]['scored'][-10:]))
            a_avg_c.append(np.mean(team_stats[at]['conceded'][-10:]))
            a_form.append(sum(team_stats[at]['pts'][-5:]))
        else:
            a_avg_s.append(1.0); a_avg_c.append(1.0); a_form.append(0)

        if pd.notna(row['home_score']) and pd.notna(row['away_score']):
            hs, ast = float(row['home_score']), float(row['away_score'])
            elo_system.update_rating(ht, at, hs, ast)
            
            team_stats[ht]['scored'].append(hs)
            team_stats[ht]['conceded'].append(ast)
            team_stats[at]['scored'].append(ast)
            team_stats[at]['conceded'].append(hs)
            
            if hs > ast:
                team_stats[ht]['pts'].append(3); team_stats[at]['pts'].append(0)
            elif hs < ast:
                team_stats[ht]['pts'].append(0); team_stats[at]['pts'].append(3)
            else:
                team_stats[ht]['pts'].append(1); team_stats[at]['pts'].append(1)

    df['home_elo'] = home_elos
    df['away_elo'] = away_elos
    df['elo_diff'] = df['home_elo'] - df['away_elo']
    df['home_avg_scored'] = h_avg_s
    df['home_avg_conceded'] = h_avg_c
    df['away_avg_scored'] = a_avg_s
    df['away_avg_conceded'] = a_avg_c
    df['home_form_5'] = h_form
    df['away_form_5'] = a_form

    mask_2026_wc = (df['date'].dt.year == 2026) & (df['tournament'] == 'FIFA World Cup')
    
    df_train = df[~mask_2026_wc].dropna(subset=['home_score', 'away_score']).copy()
    df_test = df[mask_2026_wc].copy()
    
    print(f"\n✅ 피처 계산 완료! 훈련 데이터: {len(df_train):,}건 / 타겟 경기: {len(df_test):,}건\n")

    # ---------------------------------------------------------
    # 💡 [예측 모드 설정] 최종 제출물 뽑을 때는 IS_TEST = False로 변경하세요!
    # ---------------------------------------------------------
    IS_TEST = True  
    
    if IS_TEST:
        print("⚠️ 테스트 모드로 동작합니다. (낮은 반복 횟수로 빠른 검증)")
        OPTUNA_TRIALS = 3
        MONTE_CARLO_SIMS = 100
    else:
        print("🔥 고정밀 예측 모드로 동작합니다. (시간이 소요될 수 있습니다)")
        OPTUNA_TRIALS = 50
        MONTE_CARLO_SIMS = 10000

    # 모델 학습 및 예측 생성
    lgb_home, lgb_away, feature_cols = train_lgbm_models(df_train, n_trials=OPTUNA_TRIALS)
    df_predictions = predict_tournament(df_test, lgb_home, lgb_away, feature_cols, n_simulations=MONTE_CARLO_SIMS)

    # ==========================================
    # 5. 후처리 포맷팅 작업 (update_format 기능 결합)
    # ==========================================
    print("\n🛠️ 데이터 포맷 최적화 및 표준화 작업 중...")
    
    # 팀 이름 표준화 규칙 적용
    name_corrections = {
        'Cura?ao': 'Curaçao',
        'Curacao': 'Curaçao',
        'DR Congo': 'Congo DR',
        'Cape Verde': 'Cape Verde Islands',
        'Czech Republic': 'Czechia',
        'Bosnia and Herzegovina': 'Bosnia-Herzegovina'
    }
    df_predictions['team1'] = df_predictions['team1'].replace(name_corrections)
    df_predictions['team2'] = df_predictions['team2'].replace(name_corrections)

    # type 컬럼 형식 변경 ('group' -> 'Group Stage')
    if 'type' in df_predictions.columns:
        df_predictions['type'] = df_predictions['type'].astype(str).str.strip().replace({'group': 'Group Stage'})

    # 필요한 컬럼만 추출 및 순서 보장
    desired_columns = ['team1', 'team2', 'team1_score', 'team2_score', 'team1_prob', 'team2_prob', 'type']
    columns_to_keep = [col for col in desired_columns if col in df_predictions.columns]
    df_predictions = df_predictions[columns_to_keep]

    # UTF-8 (BOM 포함)으로 최종 파일 저장 (엑셀 깨짐 방지)
    df_predictions.to_csv(output_path, index=False, encoding='utf-8-sig')
    
    print("\n==========================================")
    print(f"🎉 모든 작업 완료! 포맷 변환이 끝난 최종 예측 파일이 저장되었습니다.")
    print(f"💾 저장 파일명: {output_path}")
    print("==========================================")
    print(df_predictions.head())

/opt/miniconda3/envs/facamp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 월드컵 대진표 예측 및 포맷 변환 통합 파이프라인 시작...


✅ 피처 계산 완료! 훈련 데이터: 49,405건 / 타겟 경기: 72건

⚠️ 테스트 모드로 동작합니다. (낮은 반복 횟수로 빠른 검증)

[1/2] 홈팀 득점 예측 모델 튜닝 중...

[2/2] 원정팀 득점 예측 모델 튜닝 중...

🏃‍♂️ 각 경기당 100번의 몬테카를로 시뮬레이션을 실행합니다...


100%|██████████| 72/72 [00:00<00:00, 3819.22it/s]


🛠️ 데이터 포맷 최적화 및 표준화 작업 중...

🎉 모든 작업 완료! 포맷 변환이 끝난 최종 예측 파일이 저장되었습니다.
💾 저장 파일명: submission_consensus_formatted.csv
           team1               team2  team1_score  team2_score  team1_prob  \
0    South Korea             Czechia            2            1      0.6806   
1         Mexico        South Africa            2            0      0.9518   
2         Canada  Bosnia-Herzegovina            3            0      0.9333   
3  United States            Paraguay            2            1      0.5556   
4          Qatar         Switzerland            0            1      0.1098   

   team2_prob         type  
0      0.3194  Group Stage  
1      0.0482  Group Stage  
2      0.0667  Group Stage  
3      0.4444  Group Stage  
4      0.8902  Group Stage  
